7.1 Start with a baseline

Before building any ML model, create naive baselines to beat: (1) season average attendance, (2) same fixture last year, (3) average by opponent category. If your model cannot beat these, you need to revisit your features.

In [1]:
import pandas as pd

matches = pd.read_csv("data/gold_match_with_league_standings.csv")

matches

,match_id,match_date,kickoff_time_local,match_date_utc,kickoff_time_utc,matchday,competition_id,competition_name,season,season_id,...,result_home,goals_home_ht,goals_away_ht,goals_home_ft,goals_away_ft,is_home_match,last_result_vs_opponent,tickets_scanned,league_position_home,league_position_away
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022-07-23Z,16:15:00Z,1.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,L,0,0,0,2,False,L 1-2,NaN,14,9
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022-07-30Z,16:15:00Z,2.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,W,1,0,2,0,True,NaN,5565.0,1,1
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,2022-08-07Z,19:00:00Z,3.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,W,2,1,4,2,False,L 0-1,NaN,2,1
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,2022-08-14Z,16:30:00Z,4.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,L,0,2,0,3,True,L 1-4,7440.0,4,8
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,2022-08-21Z,16:30:00Z,5.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,L,0,2,1,3,False,W 2-1,NaN,12,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,esy974nh0dqx61hrn6g35y044,2026-02-07,20:45:00,2026-02-07Z,19:45:00Z,24.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,L,0,2,1,3,False,W 4-0,NaN,5,14
138,ewgb5fczpfa71pfzihlip6yok,2026-02-14,16:00:00,2026-02-14Z,15:00:00Z,25.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,W,2,1,3,2,True,W 1-0,5137.0,13,16
139,exzov7o3p9v5vqr86qipfm1w4,2026-02-21,18:15:00,2026-02-21Z,17:15:00Z,26.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,W,0,0,2,1,False,L 0-1,NaN,3,13
140,f19ucr3rev3uy5uvxjxznqsk4,2026-02-28,20:45:00,2026-02-28Z,19:45:00Z,27.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,W,1,1,5,1,False,D 1-1,NaN,4,13


In [5]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

match = pd.read_csv("data/gold_match.csv")
tickets = pd.read_csv("data/gold_match_tickets.csv")
context = pd.read_csv("data/gold_match_context.csv")

match = match[match["is_home_match"] == True]

df = match.merge(tickets, on="match_id", how="left")
df = df.merge(context, on="match_id", how="left")

if "match_date_x" in df.columns:
    df = df.rename(columns={"match_date_x": "match_date"})
elif "match_date_y" in df.columns:
    df = df.rename(columns={"match_date_y": "match_date"})

df = df.sort_values(["away_team", "match_date"])

df["season_avg"] = df.groupby("season")["tickets_scanned"].transform("mean")
mae_season = mean_absolute_error(df["tickets_scanned"], df["season_avg"])

df["fixture_last_year"] = df.groupby("away_team")["tickets_scanned"].shift(1)
df_fixture = df.dropna(subset=["fixture_last_year"])
mae_fixture = mean_absolute_error(
    df_fixture["tickets_scanned"],
    df_fixture["fixture_last_year"]
)

df["opponent_avg"] = df.groupby("away_team")["tickets_scanned"].transform("mean")
mae_opponent = mean_absolute_error(df["tickets_scanned"], df["opponent_avg"])

print("Baseline 1 - Season Average MAE:", mae_season)
print("Baseline 2 - Same Fixture MAE:", mae_fixture)
print("Baseline 3 - Opponent Average MAE:", mae_opponent)

Baseline 1 - Season Average MAE: 1398.8086637471893
Baseline 2 - Same Fixture MAE: 2220.64
Baseline 3 - Opponent Average MAE: 1249.7046948356808
